## Where to put API keys

In Google Colab, open the left sidebar and use **Secrets** to store your credentials. Use these exact secret names so the notebook can read them automatically:

- `ZILLIZ_URI`
- `ZILLIZ_TOKEN`
- `MONGODB_URL`
- `MONGODB_DB`
- `NVIDIA_NIM_API_KEY`
- `HF_API_TOKEN`
- `COHERE_API_KEY`
- `NCBI_API_KEY`
- `NCBI_EMAIL`

Optional settings you can also add as secrets if you want to override the defaults:

- `VECTOR_COLLECTION_V2`
- `VECTOR_COLLECTION`
- `VECTOR_DIM`
- `EMBED_PROVIDER`
- `RERANK_PROVIDER`
- `DENSE_MODEL_NAME`
- `RERANKER_MODEL_NAME`
- `PARSING_THREAD_WORKERS`
- `INGESTION_THREAD_WORKERS`

## How to run it

1. Open the notebook in Colab and connect to a runtime with GPU enabled.
2. Add the secrets above in the Colab Secrets panel.
3. Update `DATA_DIR` in the configuration cell to point to your mounted Drive or uploaded files.
4. Run the cells from top to bottom.
5. The notebook will install dependencies, configure the repo settings, start GROBID, run ingestion, and verify the stored data.

## Notes

- `EMBED_PROVIDER=local` and `RERANK_PROVIDER=local` are for Colab GPU ingestion.
- If you want CPU-only query serving later, use the remote provider settings in the repo for the query side.
- Save any persistent outputs under `/content/drive` or another mounted location, not inside temporary runtime folders.


# OpenInsight — Colab Ingestion (CPU-Safe)

**Data sources:** PubMed XMLs + StatPearls (via PubMed)  
**Parser:** `_parse_pubmed_xml_file()` — no GROBID needed for XML  
**Embedder:** HuggingFace Inference API (no GPU required)  
**Vector DB:** Zilliz Cloud  
**Document store:** MongoDB Atlas

> **Add secrets first:** Colab sidebar → 🔑 Secrets → add each key listed in Cell 3

In [1]:
# ── CELL 1: Install dependencies ──────────────────────────────────────────
import subprocess
subprocess.run(["apt-get", "install", "-qq", "-y", "tesseract-ocr"], capture_output=True)

%pip install -q \
    fastapi uvicorn python-dotenv "pydantic>=2.0" pydantic-settings \
    pymongo motor pypdf pdfplumber beautifulsoup4 requests httpx lxml biopython \
    sentence-transformers transformers "torch>=2.0" "numpy<2.0" \
    pymilvus langchain langchain-community openai redis \
    tqdm loguru tenacity python-slugify pytesseract Pillow scispacy \
    certifi cohere huggingface_hub typer

print("\n✅ Dependencies installed.")


✅ Dependencies installed.


In [2]:
# ── CELL 2: Clone / update repo ───────────────────────────────────────────
import sys
from pathlib import Path

REPO_DIR = Path('/content/openinsight')

if REPO_DIR.exists():
    %cd /content/openinsight
    !git pull origin restruct -q
    print("✅ Repo updated.")
else:
    !git clone -b restruct https://github.com/Adi103-ETAI/openinsight.git /content/openinsight -q
    print("✅ Repo cloned.")

%cd /content/openinsight
sys.path.insert(0, '/content/openinsight')
print(f"Working dir: {Path.cwd()}")

/content/openinsight
✅ Repo updated.
/content/openinsight
Working dir: /content/openinsight


In [3]:
# ── CELL 3: Configuration ─────────────────────────────────────────────────
# Required secrets to add in Colab sidebar:
#   ZILLIZ_URI, ZILLIZ_TOKEN, MONGODB_URL, NVIDIA_NIM_API_KEY,
#   HF_API_TOKEN, COHERE_API_KEY, NCBI_API_KEY, NCBI_EMAIL
import os
from pathlib import Path
from google.colab import userdata

def _s(name, default=''):
    try:
        return userdata.get(name) or default
    except Exception:
        return default

ZILLIZ_URI         = _s('ZILLIZ_URI')
ZILLIZ_TOKEN       = _s('ZILLIZ_TOKEN')
MONGODB_URL        = _s('MONGODB_URL')
MONGODB_DB         = _s('MONGODB_DB', 'openinsight')
NVIDIA_NIM_API_KEY = _s('NVIDIA_NIM_API_KEY')
HF_API_TOKEN       = _s('HF_API_TOKEN')
COHERE_API_KEY     = _s('COHERE_API_KEY')
NCBI_API_KEY       = _s('NCBI_API_KEY')
NCBI_EMAIL         = _s('NCBI_EMAIL', 'sentarc.ai@gmail.com')

# Set environment variables for the pipeline
os.environ['MONGODB_URL']           = MONGODB_URL
os.environ['MONGODB_DB']            = MONGODB_DB
os.environ['VECTOR_URI']            = ZILLIZ_URI
os.environ['VECTOR_TOKEN']          = ZILLIZ_TOKEN
os.environ['MILVUS_CLOUD']          = 'true'
os.environ['VECTOR_COLLECTION_V2']  = 'openinsight_v2'
os.environ['NVIDIA_NIM_API_KEY']    = NVIDIA_NIM_API_KEY
os.environ['EMBED_PROVIDER']        = 'huggingface'
os.environ['HF_API_TOKEN']          = HF_API_TOKEN
os.environ['HF_EMBED_MODEL']        = 'pritamdeka/S-PubMedBert-MS-MARCO'
os.environ['RERANK_PROVIDER']       = 'cohere'
os.environ['COHERE_API_KEY']        = COHERE_API_KEY
os.environ['DENSE_MODEL_NAME']      = 'pritamdeka/S-PubMedBert-MS-MARCO'
os.environ['RERANKER_MODEL_NAME']   = 'BAAI/bge-reranker-v2-m3'
os.environ['NCBI_API_KEY']          = NCBI_API_KEY
os.environ['NCBI_EMAIL']            = NCBI_EMAIL
os.environ['SSL_CERT_FILE']         = __import__('certifi').where()

DATA_DIR    = '/content/pubmed_xml'
SOURCE_TYPE = 'pubmed'
Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

print("✅ Config ready.")
print(f"   Embed mode  : huggingface (Inference API)")
print(f"   Data dir    : {DATA_DIR}")

✅ Config ready.
   Embed mode  : huggingface (Inference API)
   Data dir    : /content/pubmed_xml


In [4]:
import pymongo
import certifi
from urllib.parse import quote_plus
from google.colab import userdata
from pymilvus import MilvusClient
import os

# 1. MongoDB Connection Fix
def get_fixed_mongo_url():
    raw_url = userdata.get('MONGODB_URL')
    password = userdata.get('MONGODB_PASS')
    if not password:
        return raw_url
    # Use quote_plus to handle symbols in the password like @, #, !, $
    safe_pass = quote_plus(password)
    if '<db_password>' in raw_url:
        return raw_url.replace('<db_password>', safe_pass)
    return raw_url

fixed_url = get_fixed_mongo_url()
os.environ['MONGODB_URL'] = fixed_url

print("--- Diagnostic Connectivity Test ---")
try:
    # We explicitly add authSource=admin to ensure it checks credentials against the right system
    mc = pymongo.MongoClient(
        fixed_url,
        tlsCAFile=certifi.where(),
        serverSelectionTimeoutMS=5000,
        authSource='admin'
    )
    # The ping command is the 'moment of truth' for credentials
    mc.admin.command('ping')
    print("✅ MongoDB: Connected & Authenticated!")
except pymongo.errors.OperationFailure as e:
    print(f"❌ MongoDB Auth Failed: {e.details.get('errmsg')}")
    print("TIP: Log into MongoDB Atlas -> Database Access. Ensure user 'openinsight' has 'Read and write to any database' permissions.")
except Exception as e:
    print(f"❌ MongoDB General Error: {e}")

# 2. Zilliz Connection Check
try:
    zc = MilvusClient(uri=userdata.get('ZILLIZ_URI'), token=userdata.get('ZILLIZ_TOKEN'))
    cols = zc.list_collections()
    print(f"✅ Zilliz: Connected! Collections: {cols}")
except Exception as e:
    print(f"❌ Zilliz: Failed. {e}")

--- Diagnostic Connectivity Test ---
✅ MongoDB: Connected & Authenticated!
✅ Zilliz: Connected! Collections: ['openinsight_v2']


In [5]:
# ── CELL 5: Download PubMed + StatPearls XMLs ─────────────────────────────
import os, time
from pathlib import Path
from Bio import Entrez

Entrez.email = os.environ['NCBI_EMAIL']
Entrez.api_key = os.environ.get('NCBI_API_KEY') or None
SLEEP = 0.11 if Entrez.api_key else 0.35

QUERIES = [
    'StatPearls[book] diabetes mellitus',
    'StatPearls[book] hypertension management',
    'StatPearls[book] sepsis management',
    'tuberculosis drug resistant India treatment 2023',
    'snakebite envenomation India management protocol'
]

out_dir = Path(DATA_DIR)
saved = 0

for query in QUERIES:
    safe = query[:55].replace(' ', '_').replace('[', '').replace(']', '').replace('/', '-')
    fpath = out_dir / f"{safe}.xml"
    try:
        with Entrez.esearch(db='pubmed', term=query, retmax=10) as h:
            pmids = Entrez.read(h)['IdList']
        if pmids:
            with Entrez.efetch(db='pubmed', id=','.join(pmids), rettype='xml', retmode='xml') as h:
                data = h.read()
            with open(fpath, 'wb') as f:
                f.write(data if isinstance(data, bytes) else data.encode())
            saved += 1
            print(f"  ✅ Saved: {fpath.name}")
        time.sleep(SLEEP)
    except Exception as e:
        print(f"  ❌ {query[:30]}: {e}")

print(f"\nDone. Total XML files ready: {len(list(out_dir.glob('*.xml')))}")

  ✅ Saved: StatPearlsbook_diabetes_mellitus.xml
  ✅ Saved: StatPearlsbook_hypertension_management.xml
  ✅ Saved: StatPearlsbook_sepsis_management.xml
  ✅ Saved: tuberculosis_drug_resistant_India_treatment_2023.xml
  ✅ Saved: snakebite_envenomation_India_management_protocol.xml

Done. Total XML files ready: 5


In [6]:
# ── CELL 6: Run ingestion pipeline ────────────────────────────────────────
from src.ingestion.pipeline import IngestionPipeline

pipeline = IngestionPipeline()

summary = await pipeline.ingest_directory(
    directory=DATA_DIR,
    source='pubmed',
    recreate_index=False,
    batch_size=5,
    resume=True
)

print("\nINGESTION COMPLETE")
print(summary)

2026-05-16 12:17:33.099 | INFO     | src.ml.embedding.embedder:create_embedder:402 - [EmbedderFactory] Creating HuggingFaceEmbedder with model=pritamdeka/S-PubMedBert-MS-MARCO
2026-05-16 12:17:33.100 | INFO     | src.ml.embedding.embedder:__init__:219 - [HFEmbedder] Initialized with model=pritamdeka/S-PubMedBert-MS-MARCO
2026-05-16 12:17:44.836 | WARNING  | src.ml.embedding.embedder:embed_batch:250 - [HFEmbedder] Failed to embed text 0: Client error '404 Not Found' for url 'https://api-inference.huggingface.co/pipeline/feature-extraction/pritamdeka/S-PubMedBert-MS-MARCO'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404
2026-05-16 12:17:45.090 | WARNING  | src.ml.embedding.embedder:embed_batch:250 - [HFEmbedder] Failed to embed text 1: Client error '404 Not Found' for url 'https://api-inference.huggingface.co/pipeline/feature-extraction/pritamdeka/S-PubMedBert-MS-MARCO'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/


INGESTION COMPLETE
{'files_total': 5, 'files_parsed': 20, 'documents_stored': 20, 'chunks_created': 42, 'chunks_indexed': 42, 'files_failed': -15, 'chunks_deduped': 0, 'chunks_quality_filtered': 0}
